# 05A - Generating Context AwareQA

In [1]:
import json
import re
import time
import hashlib
import datetime
import logging
import os
from collections import defaultdict
from openai import OpenAI
from tqdm.auto import tqdm

## 1. Konfigurasi

In [2]:
CONFIG = {
    "input_dir":               "/kaggle/input/notebooks/muhammadnabilul/04-chunking-and-splitting/chunks",
    "output_dir":              "output_05a_v2",
    "llm_model":               "deepseek-v4-flash",
    "llm_temperature_qa":      0.7,
    "llm_temperature_summary": 0.3,
    "max_completion_summary":  1100,
    "max_tokens_per_context":  1500,
    "overlap_tokens":          150,
    "dataset_version":         "v2.0",
    "retry_attempts":          3,
    "retry_delay":             2.0,
    "request_delay":           1.0,
    "checkpoint_interval":     100,
    "preview_interval":        10,
    "min_section_tokens":      100,
    "max_sections_per_doc":    23,
    "skip_section_sources":    {"lampiran", "prolog"},
    "test_mode":               False,
    "max_documents_test":      None,
    "span_min_match_ratio":    0.85,
    "max_context_spans":       3,
 
    "system_prompt_qa": (
        "Anda adalah generator dataset instruction-response ter-grounding untuk melatih Small "
        "Language Model (SLM) sebagai komponen generator pada sistem Retrieval-Augmented Generation "
        "(RAG) publikasi statistik BPS.\n\n"
        "TUJUAN\n"
        "Buat interaction QA yang melatih model memilih kutipan relevan (context_spans), "
        "menghubungkan evidensi dengan jawaban secara ringkas (cot), lalu menjawab hanya berdasarkan "
        "Dokumen Referensi (response). Ini mensimulasikan kondisi RAG nyata: retriever mengembalikan "
        "potongan chunk yang tidak selalu berurutan, dan model harus memilih bagian relevan sebelum "
        "menjawab. Setiap interaction wajib membahas sub-topik berbeda, dan instruction wajib tidak "
        "dapat dijawab secara andal tanpa membaca span yang dipilih.\n\n"
        "PROSEDUR (lakukan secara internal untuk setiap interaction, jangan tampilkan prosesnya)\n"
        "TAHAP 1 — Seleksi sub-topik: pilih sub-topik yang berbeda dari interaction lain, berasal dari "
        "narasi Dokumen Referensi, memerlukan pembacaan dokumen (bukan pengetahuan umum).\n"
        "TAHAP 2 — Seleksi evidensi: pilih 1-3 span kutipan PERSIS dari dokumen (angka, ejaan, tanda "
        "baca, istilah teknis tidak boleh diubah). Span harus satu kalimat atau beberapa kalimat "
        "berurutan yang koheren — jangan memotong kalimat, menggabungkan lokasi terpisah, atau "
        "mencampur topik (indikasi artefak linearisasi tabel/grafik: kalimat melompat topik tanpa "
        "transisi wajar, angka muncul tanpa satuan/konteks jelas). Abaikan dan JANGAN JADIKAN SPAN "
        "baris/fragmen dengan ciri-ciri berikut, karena itu SELALU residu tabel yang ter-linearisasi, "
        "BUKAN kalimat naratif, walau tampak mengandung angka yang benar:\n"
        "  (a) baris yang didominasi karakter pipe ('|');\n"
        "  (b) tanda kurung siku angka seperti '[1]', '[2]' (superscript catatan kaki tabel);\n"
        "  (c) sebuah frasa/label (nama indikator, nama baris tabel) yang LANGSUNG diikuti dua atau "
        "lebih angka berurutan TANPA kata kerja atau kata penghubung gramatikal di antaranya — "
        "contoh pola terlarang: 'Persentase Pekerja Formal [1] % 64,80 63,68 -1,12' atau "
        "'Rata-Rata Upah per Bulan [1] rupiah 4.056.847 4.155.520 98.673'. Ini adalah baris tabel "
        "(label, satuan, nilai tahun-1, nilai tahun-2, selisih) yang dibaca berurutan oleh mesin "
        "ekstraksi PDF, BUKAN kalimat yang ditulis manusia.\n"
        "Jika angka yang relevan HANYA muncul dalam pola semacam ini (tanpa kalimat naratif yang "
        "menjelaskannya secara gramatikal lengkap), JANGAN gunakan angka tersebut sama sekali "
        "walau tergoda karena angkanya tampak berguna — batalkan kandidat ini dan pilih sub-topik "
        "lain yang evidensinya berupa kalimat utuh. Jangan gunakan isi tabel atau nilai visual "
        "sebagai dasar dalam bentuk apapun.\n"
        "TAHAP 3 — Pembentukan instruction: spesifik dan natural. Instruction dapat berupa pertanyaan, "
        "permintaan penjelasan, perbandingan, uraian, atau navigasi lokasi pembahasan, selama type "
        "tersedia dalam daftar yang diizinkan pada pesan pengguna. Untuk factual_retrieval, "
        "trend_interpretation, comparative_analysis: masukkan angka sumber spesifik jika ada dalam "
        "span (bukan deskripsi kualitatif). Angka turunan (selisih/pertumbuhan/rata-rata) HANYA boleh "
        "jika KEDUA angka sumber tampil eksplisit di span — jika tidak, pilih sub-topik lain.\n"
        "TAHAP 4 — Pembentukan cot: maksimal dua kalimat merangkum evidensi dan hubungan langsungnya "
        "dengan jawaban. Pola wajib: 'Teks menyatakan \"<kutipan singkat>\". Karena itu, <kesimpulan "
        "yang langsung didukung>.' Jangan menambah penalaran atau fakta di luar span.\n"
        "TAHAP 5 — Pembentukan response: jawaban final yang berdiri sendiri, hanya berisi fakta yang "
        "didukung span. Jangan menyebut wadah retrieval ('konteks ini', 'teks sumber', 'dokumen ini', "
        "'passages', 'di atas'). Atribusi ke entitas nyata (mis. 'menurut Susenas 2024') opsional. "
        "Jangan menambah asumsi, estimasi, ketidakpastian, atau catatan proses dalam tanda kurung — "
        "jika suatu klaim tidak dapat dipastikan, jangan sertakan sama sekali. Pertahankan istilah "
        "teknis BPS persis (Susenas, TPT, IPM, PDRB, garis kemiskinan, dll).\n"
        "TAHAP 6 — Validasi silang internal, periksa sebelum mengeluarkan object:\n"
        "  A. Semua span ditemukan persis (verbatim) dalam Dokumen Referensi.\n"
        "  B. Response tidak melampaui informasi pada span.\n"
        "  C. Angka, tahun, satuan, dan nama wilayah konsisten antara span dan response.\n"
        "  D. Type termasuk dalam daftar yang diizinkan.\n"
        "  E. Sub-topik belum dipakai interaction lain dalam batch ini.\n"
        "  F. Untuk source_navigation: jawaban hanya menyatakan LETAK pembahasan, bukan isi tabel "
        "atau nilai visual.\n"
        "  Jika salah satu gagal, revisi object atau ganti kandidat sebelum output.\n\n"
        "PRIORITAS SAAT KONFLIK\n"
        "Jika aturan-aturan di atas bertentangan dalam satu kasus, dahulukan berurutan: "
        "(1) grounding pada dokumen, (2) validitas JSON, (3) kutipan verbatim, "
        "(4) larangan tabel/visual/artefak, (5) keunikan sub-topik, (6) kelengkapan gaya jawaban. "
        "Jika jumlah interaction yang diminta tidak dapat dipenuhi tanpa melanggar grounding, "
        "keluarkan HANYA interaction yang valid — JANGAN mengarang fakta demi memenuhi kuota.\n\n"
        "CONTOH (BENAR vs SALAH)\n"
        "[TAHAP 2 — span tidak koheren]\n"
        "SALAH: context_spans=['tingkat penyuapan yang dialami oleh (15,46 persen), angka tersebut "
        "kembali bisnis'] — dua kalimat berbeda tercampur, topik melompat tanpa transisi.\n"
        "BENAR: context_spans=['Persentase masyarakat yang melaporkan membayar melebihi ketentuan "
        "resmi sempat mencapai titik terendah pada tahun 2022 (15,46 persen), angka tersebut kembali "
        "meningkat pada tahun 2023 (16,33 persen).']\n"
        "[TAHAP 2 — residu tabel, aturan (c)]\n"
        "SALAH: context_spans=['Persentase Pekerja Formal [1] % 64,80 63,68 -1,12'] — label diikuti "
        "deret angka tanpa kalimat, itu baris tabel bukan narasi, walau angkanya tampak berguna.\n"
        "BENAR (jika kalimat naratifnya memang ada di dokumen): context_spans=['Persentase pekerja "
        "formal tercatat sebesar 64,80 persen pada tahun 2021 dan menurun menjadi 63,68 persen pada "
        "tahun 2022.'] — angka sama, tapi dikutip dari kalimat naratif utuh, bukan baris tabel.\n"
        "BENAR (jika hanya ada versi baris tabel, tidak ada narasi): batalkan sub-topik ini, pilih "
        "sub-topik lain yang punya evidensi berupa kalimat naratif utuh.\n"
        "[TAHAP 4 — cot]\n"
        "BENAR: cot='Teks menyatakan \"titik terendah pada tahun 2022 (15,46 persen)\". Karena itu, "
        "angka pada 2022 adalah 15,46 persen.'\n"
        "[TAHAP 3 — angka wajib eksplisit]\n"
        "SALAH: instruction='Bagaimana kondisi ketenagakerjaan Jabar?' "
        "response='Ketenagakerjaan Jabar mengalami peningkatan yang cukup baik.'\n"
        "BENAR: instruction='Berapa Tingkat Pengangguran Terbuka (TPT) Jawa Barat?' "
        "response='TPT Jawa Barat tercatat sebesar 7,44 persen.'\n"
        "[TAHAP 3 — angka turunan]\n"
        "SALAH: response='...turun dari 72,60 poin (tidak disebut eksplisit, dihitung dari "
        "58,68+13,92) menjadi 58,68 poin.'\n"
        "BENAR: response='IPM DKI Jakarta 81,65 dan IPM Papua 61,39, selisih keduanya 20,26 poin.'\n"
        "[TAHAP 6.F — source_navigation]\n"
        "BENAR: instruction='Di bagian mana pembahasan mengenai garis kemiskinan perkotaan dapat "
        "ditemukan?' response='Pembahasan garis kemiskinan perkotaan terdapat pada subbab Kemiskinan, "
        "dirujuk melalui Tabel 3.2.'\n\n"
        "FORMAT OUTPUT\n"
        "Keluarkan hanya JSON array valid, tanpa markdown, tanpa komentar, tanpa kalimat pembuka.\n"
        "[\n"
        "  {\n"
        "    \"task\": \"qa\",\n"
        "    \"type\": \"<type dari daftar yang diizinkan>\",\n"
        "    \"context_spans\": [\"<kutipan verbatim 1>\", \"<kutipan verbatim 2 (opsional)>\"],\n"
        "    \"instruction\": \"<permintaan spesifik>\",\n"
        "    \"cot\": \"<ringkasan evidensi maksimal dua kalimat>\",\n"
        "    \"response\": \"<jawaban final mandiri>\",\n"
        "    \"reasoning_source\": \"<single_paragraph|multi_paragraph|whole_section>\",\n"
        "    \"context_focus\": \"<sub-topik singkat>\"\n"
        "  }\n"
        "]"
    ),
 
    "system_prompt_summary": (
        "Anda adalah generator dataset instruction-response ter-grounding untuk melatih Small "
        "Language Model (SLM) sebagai komponen generator pada sistem RAG publikasi statistik BPS.\n\n"
        "TUJUAN\n"
        "Buat tepat satu pasangan interaction-response bertipe 'document_overview' berdasarkan hanya "
        "pada Dokumen Referensi yang diberikan. Pasangan ini melatih SLM memilih span relevan dari "
        "beberapa bagian dokumen, bernalar ringkas via cot, lalu mensintesis jadi satu gambaran "
        "menyeluruh.\n\n"
        "PROSEDUR (lakukan secara internal, jangan tampilkan proses perencanaan)\n"
        "A. Cakupan: baca seluruh Dokumen Referensi dari awal sampai akhir. Tentukan poin penting "
        "dari bagian awal, tengah, dan akhir. Abaikan baris yang didominasi karakter pipe ('|') "
        "karena merupakan artefak linearisasi tabel — jangan jadikan dasar fakta atau angka.\n"
        "B. Seleksi evidensi: pilih 2-3 span kutipan VERBATIM dari bagian berbeda (awal, tengah, "
        "akhir) yang koheren secara sintaksis. Jika suatu bagian tidak koheren atau tercampur topik "
        "(artefak linearisasi tabel/grafik), lewati bagian itu dan pilih span dari bagian lain yang "
        "bersih. Pilih sekurangnya dua indikator atau temuan spesifik yang dinyatakan eksplisit dalam "
        "sumber, BILA TERSEDIA — jika Dokumen Referensi tidak memiliki sedikitnya dua indikator "
        "eksplisit, tetap buat sintesis paling informatif yang didukung sumber tanpa mengarang fakta, "
        "dan JANGAN mengklaim terdapat dua indikator jika sebenarnya hanya satu.\n"
        "C. Pembentukan instruction: rumuskan permintaan gambaran menyeluruh yang natural dan "
        "spesifik terhadap topik, wilayah, atau periode yang tersedia. Jangan gunakan permintaan "
        "generik seperti 'Apa isi dokumen ini?'.\n"
        "D. Pembentukan cot: rangkai kutipan singkat dari MASING-MASING span sebelum menyimpulkan "
        "sintesis akhir. Pola: 'Bagian awal menyebutkan \"<kutipan>\"; bagian lain menyebutkan "
        "\"<kutipan>\". Sintesis: ...'. Jangan menambah fakta di luar span.\n"
        "E. Pembentukan response: gabungkan poin terpilih menjadi sintesis final yang koheren, bukan "
        "daftar fragmen terpisah. Gunakan hanya fakta yang eksplisit dalam sumber — jangan menghitung, "
        "menebak, mengisi kekosongan, atau menambahkan pengetahuan eksternal. Jangan menyebut wadah "
        "retrieval ('dokumen ini', 'bagian ini', 'passages', 'teks sumber ini', 'di atas'). Atribusi "
        "ke entitas nyata (mis. BPS) boleh digunakan hanya jika didukung sumber. Jangan menulis "
        "catatan proses, asumsi, estimasi, atau keterangan ketidakpastian dalam tanda kurung — jika "
        "suatu fakta tidak dapat dipastikan, hilangkan fakta itu, jangan disebutkan sama sekali.\n\n"
        "PEMERIKSAAN SEBELUM OUTPUT\n"
        "- Keluaran berisi tepat satu object.\n"
        "- Setiap fakta response dapat ditelusuri persis ke context_spans.\n"
        "- Response mencakup informasi dari awal, tengah, dan akhir sumber jika ketiga bagian "
        "memuat informasi naratif yang koheren.\n"
        "- Tidak ada baris pipe atau artefak tabel yang dipakai sebagai fakta.\n"
        "- Tidak ada frasa meta-talk atau wadah retrieval yang terlarang.\n"
        "- JSON valid, tidak ada teks di luar array.\n\n"
        "CONTOH (BENAR vs SALAH)\n"
        "SALAH (generik, tidak sintesis): instruction='Apa isi dokumen ini?' "
        "response='Dokumen ini membahas ketenagakerjaan secara umum.'\n"
        "BENAR: instruction='Uraikan temuan utama mengenai ketenagakerjaan Jawa Barat tahun 2024.' "
        "cot='Bagian awal menyebutkan \"TPT Jawa Barat tercatat 7,44 persen\"; bagian akhir menyebutkan "
        "\"tingkat partisipasi angkatan kerja tercatat 67,2 persen\". Sintesis: kedua indikator "
        "menunjukkan pemulihan bertahap.' response='TPT Jawa Barat tercatat 7,44 persen. Tingkat "
        "partisipasi angkatan kerja tercatat 67,2 persen, menunjukkan pemulihan bertahap.'\n\n"
        "FORMAT OUTPUT\n"
        "Keluarkan hanya JSON array valid, tanpa markdown, tanpa komentar, tanpa kalimat pembuka.\n"
        "[\n"
        "  {\n"
        "    \"task\": \"summary\",\n"
        "    \"type\": \"document_overview\",\n"
        "    \"context_spans\": [\"<kutipan verbatim awal>\", \"<kutipan verbatim tengah/akhir>\"],\n"
        "    \"instruction\": \"<permintaan gambaran menyeluruh yang spesifik>\",\n"
        "    \"cot\": \"<kutip tiap span singkat + sintesis>\",\n"
        "    \"response\": \"<sintesis final mencakup sumber secara menyeluruh>\",\n"
        "    \"reasoning_source\": \"whole_section\",\n"
        "    \"context_focus\": \"<topik, wilayah, dan periode bila tersedia>\"\n"
        "  }\n"
        "]"
    ),
}
 
QA_BATCHES = [
    {
        "pattern":          "literal_reading",
        "allowed_types":    "factual_retrieval, source_navigation",
        "focus": (
            "Reasoning pattern: LITERAL READING — membaca dan menemukan informasi yang tertulis "
            "eksplisit di teks sumber.\n"
            "Tipe yang diizinkan: factual_retrieval ATAU source_navigation.\n"
            "- factual_retrieval: interaction meminta fakta, angka, atau pernyataan eksplisit "
            "yang tertulis langsung di narasi.\n"
            "- source_navigation: interaction meminta LOKASI atau rujukan — di subbab/bagian mana "
            "suatu topik dibahas, atau tabel/gambar nomor berapa yang membahasnya. Menunjuk "
            "keberadaan/letak, BUKAN membaca isi sel tabel.\n"
            "Hasilkan {n_pairs} interaction. Setiap interaction membahas sub-topik BERBEDA dari "
            "bagian BERBEDA dalam narasi. Usahakan kedua tipe terwakili minimal 1 kali; sisanya bebas."
        ),
        "generation_focus": "literal_reading",
    },
    {
        "pattern":          "concept_understanding",
        "allowed_types":    "definition_clarification, methodology_explanation",
        "focus": (
            "Reasoning pattern: CONCEPT UNDERSTANDING — memahami makna istilah dan metodologi.\n"
            "Tipe yang diizinkan: definition_clarification ATAU methodology_explanation.\n"
            "- definition_clarification: interaction meminta penjelasan istilah, konsep, atau "
            "indikator statistik yang didefinisikan dalam narasi.\n"
            "- methodology_explanation: interaction meminta penjelasan cara pengukuran, metode "
            "pengumpulan data, atau formula yang dijelaskan dalam narasi.\n"
            "Hasilkan {n_pairs} interaction. Setiap interaction dari sub-topik BERBEDA. "
            "Usahakan kedua tipe terwakili minimal 1 kali; sisanya bebas."
        ),
        "generation_focus": "concept_understanding",
    },
    {
        "pattern":          "evidence_integration",
        "allowed_types":    "trend_interpretation, causal_reasoning",
        "focus": (
            "Reasoning pattern: EVIDENCE INTEGRATION — menginterpretasikan bukti dan menalar sebab-akibat.\n"
            "Tipe yang diizinkan: trend_interpretation ATAU causal_reasoning.\n"
            "- trend_interpretation: interaction meminta interpretasi pola, perubahan, atau "
            "dinamika dari data (naik/turun/stabil, mengapa berfluktuasi).\n"
            "- causal_reasoning: interaction meminta penjelasan penyebab, dampak, atau "
            "implikasi dari fenomena yang dibahas dalam narasi.\n"
            "Hasilkan {n_pairs} interaction. Berbasis bukti dari bagian BERBEDA dalam narasi. "
            "Usahakan kedua tipe terwakili minimal 1 kali; sisanya bebas."
        ),
        "generation_focus": "evidence_integration",
    },
    {
        "pattern":          "analytical_reasoning",
        "allowed_types":    "comparative_analysis, multi_source_synthesis",
        "focus": (
            "Reasoning pattern: ANALYTICAL REASONING — membandingkan dan mensintesis lintas informasi.\n"
            "Tipe yang diizinkan: comparative_analysis ATAU multi_source_synthesis.\n"
            "- comparative_analysis: interaction meminta perbandingan eksplisit antara dua atau "
            "lebih entitas (wilayah, periode, kelompok, indikator) menggunakan data di narasi.\n"
            "- multi_source_synthesis: interaction membutuhkan penggabungan informasi dari "
            "MINIMAL DUA bagian berbeda. reasoning_source WAJIB multi_paragraph atau whole_section.\n"
            "Hasilkan {n_pairs} interaction dari 2 tipe BERBEDA."
        ),
        "generation_focus": "analytical_reasoning",
    },
]
 
ALLOWED_TASKS = {"qa", "summary"}
ALLOWED_SUBTASKS = {
    "factual_retrieval", "source_navigation",
    "definition_clarification", "methodology_explanation",
    "trend_interpretation", "causal_reasoning",
    "comparative_analysis", "multi_source_synthesis",
    "document_overview",
}
 
def get_n_pairs(context_token_count):
    if context_token_count < 1200:
        return 2
    elif context_token_count <= 2200:
        return 4
    return 6
 
def get_max_completion_qa(n_pairs):
    return (n_pairs + 1) * 650
 
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("API_Deepseek")
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
 
GENERATION_TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
 
os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(f"{CONFIG['output_dir']}/logs", exist_ok=True)
 
OUT                   = CONFIG["output_dir"]
RAW_JSONL             = f"{OUT}/raw_records.jsonl"
FAILURES_JSONL        = f"{OUT}/generation_failures.jsonl"
PROCESSED_UNITS_JSONL = f"{OUT}/processed_units.jsonl"

### a. Logging

In [3]:
def setup_logger(name, filepath):
    handler = logging.FileHandler(filepath, encoding="utf-8")
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.addHandler(handler)
    return logger
 
# Logger untuk alur normal generate
gen_log  = setup_logger("generation", f"{OUT}/logs/generation.log")
# Logger cadangan khusus kegagalan (saat ini kegagalan juga ditulis ke JSONL)
fail_log = setup_logger("failure",    f"{OUT}/logs/failure.log")

### b. Checkpoint

In [4]:
def load_jsonl(filepath):
    if not os.path.exists(filepath):
        return []
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records
 
# Buka file dalam mode append (tidak menimpa data lama)
# Tulis satu record per baris
def append_jsonl(filepath, records):
    with open(filepath, "a", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())
 
# satu jendela konteks × satu jenis generate (QA-batch-0/1 atau summary)
def unit_key(document_title, section_id, batch_idx, generation_focus):
    return f"{document_title}|{section_id}|b{batch_idx}|{generation_focus}"
 
# Baca semua kunci unit yang sudah selesai dari file resume ke dalam set
def load_completed_units():
    return {r["k"] for r in load_jsonl(PROCESSED_UNITS_JSONL)}
 
# Tambahkan satu kunci ke file resume; dipanggil setelah unit selesai 
def mark_unit(key):
    append_jsonl(PROCESSED_UNITS_JSONL, [{"k": key}])

## 2. Fungsi Persiapan

### a. Chunks ke section

In [5]:
# Urutkan nama file agar urutan pemrosesan konsisten antar run
def load_chunks(input_dir):
    chunks = []
    json_files = sorted(f for f in os.listdir(input_dir) if f.endswith(".json"))
    for fname in json_files:
        fpath = os.path.join(input_dir, fname)
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            chunks.extend(data)
        else:
            chunks.append(data)
    gen_log.info(f"Loaded {len(chunks)} chunks from {len(json_files)} files")
    return chunks

In [6]:
# Membalik proses chunking: kelompokkan chunk kembali ke section induknya
# Kunci unik section: kombinasi judul dokumen + ID section
def reconstruct_sections(chunks_data):
    sections = {}
    for chunk in chunks_data:
        sec_key = (chunk.get("document_title"), chunk.get("section_id"))
        if sec_key not in sections:
            sections[sec_key] = {
                "section_title":      chunk.get("section_title"),
                "section_id":         chunk.get("section_id"),
                "section_source":     chunk.get("section_source", ""),
                "document_title":     chunk.get("document_title"),
                "document_year":      chunk.get("document_year"),
                "bps_url":            chunk.get("bps_url"),
                "section_chunks":     [],
                "section_token_est":  0,
            }
        sections[sec_key]["section_chunks"].append({
            "chunk_id":      chunk["chunk_id"],
            "text":          chunk.get("text", ""),
            "chunk_type":    chunk.get("chunk_type", "text_only"),
            "token_estimate": chunk.get("token_estimate", 0),
            "page_start":    chunk.get("chunk_page_start", 0),
            "page_end":      chunk.get("chunk_page_end", 0),
        })
        sections[sec_key]["section_token_est"] += chunk.get("token_estimate", 0)
    return sections
 
 
def _is_noise_title(title):
    # Judul kosong/None selalu dianggap noise 
    if not title:
        return True
    # Hitung karakter non-ASCII atau simbol artefak ekstraksi PDF    
    non_ascii = sum(1 for ch in title if ord(ch) > 127 or ch in "©®™|=\\")
    # Jika lebih dari 15% karakter adalah noise → judul tidak valid
    return non_ascii / max(len(title), 1) > 0.15
 
# Filter
def filter_section(sec_data):
    # Tolak: section berasal dari sumber yang selalu dilewati
    if sec_data["section_source"] in CONFIG["skip_section_sources"]:
        return False, "bad_source"
    # Tolak: section terlalu pendek untuk menghasilkan QA berkualitas    
    if sec_data["section_token_est"] < CONFIG["min_section_tokens"]:
        return False, "too_short"
    # Tolak: semua chunk hanya tabel atau gambar (tidak ada narasi teks)    
    if all(c["chunk_type"] in ("table_rich", "figure_rich") for c in sec_data["section_chunks"]):
        return False, "all_non_text"
    # Tolak: judul section mengandung terlalu banyak karakter noise PDF
    if _is_noise_title(sec_data.get("section_title", "")):
        return False, "noise_title"
    return True, "ok"
 
 
def select_sections(sections):
    # Kelompokkan per dokumen
    by_doc = defaultdict(list)
    for sec_key, sec_data in sections.items():
        lolos, reason = filter_section(sec_data)
        if lolos:
            by_doc[sec_data["document_title"]].append((sec_key, sec_data))
 
    selected = {}
    filter_stats = defaultdict(int)
 
    # Hitung yang difilter
    for sec_key, sec_data in sections.items():
        lolos, reason = filter_section(sec_data)
        if not lolos:
            filter_stats[reason] += 1
 
    cap = CONFIG["max_sections_per_doc"]
    for doc_title, sec_list in by_doc.items():
        # Prioritas: section dengan token terbanyak (paling substansial) diproses duluan
        sec_list.sort(key=lambda x: x[1]["section_token_est"], reverse=True)
        # Ambil maksimal `cap` section per dokumen untuk keseimbangan dataset
        for sec_key, sec_data in sec_list[:cap]:
            selected[sec_key] = sec_data
 
    gen_log.info(
        f"Section selection: raw={len(sections)} | "
        f"filtered={sum(filter_stats.values())} ({dict(filter_stats)}) | "
        f"selected={len(selected)}"
    )
    return selected, dict(filter_stats)

### b. Overlap Batching

In [7]:
# Estimasi kasar: jumlah kata × 1.35 (konsisten dengan Notebook 04)
def _estimate_tokens(text):
    return max(1, int(len(text.split()) * 1.35))
 
def batch_section(section_chunks, max_tokens, overlap_tokens):
    # Section kosong tidak menghasilkan batc
    if not section_chunks:
        return []
 
    step = max_tokens - overlap_tokens
 
    # Buat word-stream dengan provenance per chunk
    word_stream = []
    for sc in section_chunks:
        # Daftar nomor halaman yang dicakup chunk ini
        pages = list(range(sc["page_start"], sc["page_end"] + 1))
        for word in sc["text"].split():
            # Setiap kata membawa metadata sumbernya
            word_stream.append({"word": word, "chunk_id": sc["chunk_id"], "pages": pages})
 
    if not word_stream:
        return []
 
    # Konversi max_tokens dan step ke jumlah kata (1 token ≈ 0.74 kata)
    words_per_token = 0.74
    max_words  = int(max_tokens * words_per_token)
    step_words = int(step * words_per_token)
 
    batches = []
    pos = 0
 
    while pos < len(word_stream):
        # Ambil potongan sepanjang max_words kata
        window = word_stream[pos: pos + max_words]
 
        # Kumpulkan chunk_id unik dalam urutan kemunculan (union provenance)
        chunk_ids_seen = []
        chunk_id_set   = set()
        # Kumpulkan semua nomor halaman yang tercakup jendela ini
        pages_seen     = set()
 
        for w in window:
            if w["chunk_id"] not in chunk_id_set:
                chunk_ids_seen.append(w["chunk_id"])
                chunk_id_set.add(w["chunk_id"])
            pages_seen.update(w["pages"])
 
        # Gabungkan kata-kata jendela menjadi satu teks konteks
        context_text = " ".join(w["word"] for w in window)
        token_count  = _estimate_tokens(context_text)
 
        batches.append({
            "context_text":        context_text,
            # Daftar chunk_id sumber (untuk `retrieval_evaluable` test set)
            "context_chunk_ids":   chunk_ids_seen,
            # Halaman sumber terurut (untuk provenance sitasi)
            "source_pages":        sorted(list(pages_seen)),
            "context_token_count": token_count,
        })
 
        pos += step_words
 
    return batches

### c. json parse

In [8]:
def safe_json_parse(raw):
    raw = raw.strip()

    def _valid(result):
        return isinstance(result, list) and len(result) > 0 and all(isinstance(x, dict) for x in result)

    try:
        result = json.loads(raw)
        if _valid(result):
            return result, "direct"
        if isinstance(result, dict):
            return [result], "direct_object_wrapped"
    except json.JSONDecodeError:
        pass

    cleaned = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned).strip()
    try:
        result = json.loads(cleaned)
        if _valid(result):
            return result, "fence_stripped"
        if isinstance(result, dict):
            return [result], "fence_stripped_object_wrapped"
    except json.JSONDecodeError:
        pass

    for i, ch in enumerate(cleaned):
        if ch not in "[{":
            continue
        close_ch = "]" if ch == "[" else "}"
        for j in range(len(cleaned), i, -1):
            if cleaned[j - 1] == close_ch:
                try:
                    result = json.loads(cleaned[i:j])
                except json.JSONDecodeError:
                    continue
                if _valid(result):
                    return result, "extracted"
                if isinstance(result, dict):
                    return [result], "extracted_object_wrapped"

    return None, "failed"

### d. Generate logic

In [9]:
import asyncio
 
class BudgetExhausted(Exception):
    pass
 
 
def call_llm(system_prompt, user_content, temperature, max_completion, label=""):
    last_error = ""
    for attempt in range(1, CONFIG["retry_attempts"] + 1):
        try:
            response = client.chat.completions.create(
                model=CONFIG["llm_model"],
                temperature=temperature,
                max_tokens=max_completion,
                extra_body={"thinking": {"type": "disabled"}},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_content},
                ],
            )
            raw    = response.choices[0].message.content or ""
            finish = response.choices[0].finish_reason
            usage  = response.usage
            usage_dict = usage.model_dump() if hasattr(usage, "model_dump") else dict(usage)
            cached = (
                usage_dict.get("prompt_cache_hit_tokens")
                or usage_dict.get("prompt_tokens_cached")
                or (usage_dict.get("prompt_tokens_details") or {}).get("cached_tokens")
                or 0
            )
            if finish == "length":
                gen_log.warning(f"[{label}] output truncated")
            parsed, parse_status = safe_json_parse(raw)
            if parsed is None:
                gen_log.warning(f"[{label}] parse={parse_status} finish={finish} raw={repr(raw[:200])}")
            return parsed, parse_status, raw, {
                "prompt_tokens":     usage.prompt_tokens,
                "completion_tokens": usage.completion_tokens,
                "total_tokens":      usage.total_tokens,
                "cached_tokens":     cached,
            }
        except Exception as e:
            last_error = str(e)
            gen_log.warning(f"[{label}] attempt {attempt} error: {e}")
            print(f"  [ERR ] {label} attempt {attempt}: {str(e)[:120]}")
            if any(kw in str(e).lower() for kw in ["insufficient", "balance", "402"]):
                raise BudgetExhausted(str(e)) from e
            if attempt < CONFIG["retry_attempts"]:
                wait = CONFIG["retry_delay"] * attempt
                if "429" in str(e):
                    wait = max(wait, 60)
                    print(f"  [WAIT] rate limit, tunggu {wait:.0f}s...")
                time.sleep(wait)
    gen_log.error(f"[{label}] all attempts failed: {last_error}")
    return None, "api_error", "", {
        "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "cached_tokens": 0,
    }
 
 
async def call_llm_async(system_prompt, user_content, temperature, max_completion, label=""):
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(
        None,
        lambda: call_llm(system_prompt, user_content, temperature, max_completion, label)
    )
 
 
async def generate_all_qa_parallel(context_text, ctx_token_count, batch_label):
    n_pairs  = get_n_pairs(ctx_token_count)
    max_comp = get_max_completion_qa(n_pairs)
    sys_prompt = CONFIG["system_prompt_qa"]
    tasks = []
    for qa_cfg in QA_BATCHES:
        focus_resolved = qa_cfg["focus"].replace("{n_pairs}", str(n_pairs))
        user_content = (
            f"Dokumen Referensi:\n{context_text}\n\n"
            f"Hasilkan {n_pairs} interaction. "
            f"Reasoning pattern: {qa_cfg['pattern']}. "
            f"Type yang diizinkan: {qa_cfg['allowed_types']}.\n"
            f"{focus_resolved}"
        )
        label = f"{batch_label}|{qa_cfg['generation_focus']}"
        tasks.append(call_llm_async(sys_prompt, user_content, CONFIG["llm_temperature_qa"], max_comp, label=label))
    results = await asyncio.gather(*tasks, return_exceptions=True)
    return results, n_pairs
 
 
def generate_summary(context_text, section_title, label=""):
    user_content = f"Topik: {section_title}\n\nDokumen Referensi:\n{context_text}"
    return call_llm(
        CONFIG["system_prompt_summary"], user_content,
        CONFIG["llm_temperature_summary"], CONFIG["max_completion_summary"], label=label,
    )

### e. record ID

In [10]:
import difflib
 
def make_record_id(instruction, response, document_title, section_title, context_hash):
    payload = f"{instruction}|{response}|{document_title}|{section_title}|{context_hash}"
    return hashlib.sha256(payload.encode()).hexdigest()[:16]
 
 
def validate_context_spans(spans, window_source, min_ratio=None):
    min_ratio = min_ratio or CONFIG["span_min_match_ratio"]
    results = []
    window_norm = " ".join(window_source.split())
    for span in spans:
        span_norm = " ".join(str(span).split())
        if not span_norm:
            results.append((span, False, 0.0))
            continue
        if span_norm in window_norm:
            results.append((span, True, 1.0))
            continue
        matcher = difflib.SequenceMatcher(None, span_norm, window_norm)
        match = matcher.find_longest_match(0, len(span_norm), 0, len(window_norm))
        ratio = match.size / max(len(span_norm), 1)
        results.append((span, ratio >= min_ratio, round(ratio, 3)))
    return results

def check_spans_quality(spans, window_source):
    if not spans or not isinstance(spans, list):
        return False, "empty_spans"
    if len(spans) > CONFIG["max_context_spans"]:
        return False, "too_many_spans"
    validated = validate_context_spans(spans, window_source)
    invalid = [s for s, ok, r in validated if not ok]
    if invalid:
        return False, f"unverified_span_count={len(invalid)}"
    return True, None

### f. Proses batch

In [11]:
def print_preview(records, batch_label):
    if not records:
        return
    rec = records[0]
    sep = "─" * 60
    print(f"\n{sep}")
    print(f"PREVIEW  {batch_label}")
    print(f"  doc    : {rec['document_title'][:55]}")
    print(f"  section: {rec['section_title'][:55]}")
    print(f"  pattern: {rec['generation_focus']} | subtask: {rec['subtask_type']}")
    print(f"  source : {rec['reasoning_source']} | focus: {rec['context_focus']}")
    print(f"  spans  : {len(rec['context'])} span(s)")
    print(f"  Q: {rec['instruction']}")
    print(f"  CoT: {rec['cot'][:200]}{'...' if len(rec['cot']) > 200 else ''}")
    print(f"  A: {rec['response'][:300]}{'...' if len(rec['response']) > 300 else ''}")
    print(sep)
 
 
def process_batch(batch, batch_idx, num_batches, data, ctx_hash,
                  parsed, parse_status, generation_focus, usage):
    records, skipped = [], 0
    for item in parsed:
        instruction      = item.get("instruction", "").strip()
        response         = item.get("response", "").strip()
        cot              = item.get("cot", "").strip()
        context_spans    = item.get("context_spans", [])
        task_type        = item.get("task", "").strip()
        subtask_type     = item.get("type", "").strip()
        reasoning_source = item.get("reasoning_source", "").strip()
        context_focus    = item.get("context_focus", "").strip()
 
        if not instruction or not response or not cot:
            skipped += 1; continue
        if task_type not in ALLOWED_TASKS:
            skipped += 1; continue
        if subtask_type not in ALLOWED_SUBTASKS:
            gen_log.warning(f"invalid subtask={subtask_type} in {data['section_id']}|{generation_focus}")
            skipped += 1; continue
 
        ok, reason = check_spans_quality(context_spans, batch["context_text"])
        if not ok:
            gen_log.warning(f"span_reject={reason} in {data['section_id']}|{generation_focus}")
            skipped += 1; continue
 
        record_id = make_record_id(
            instruction, response, data["document_title"], data["section_title"], ctx_hash,
        )
        records.append({
            "record_id":            record_id,
            "dataset_version":      CONFIG["dataset_version"],
            "task_type":            task_type,
            "subtask_type":         subtask_type,
            "generation_focus":     generation_focus,
            "reasoning_source":     reasoning_source,
            "context_focus":        context_focus,
            "instruction":          instruction,
            "cot":                  cot,
            "response":             response,
            "context":              context_spans,
            "context_joined":       "\n\n---\n\n".join(context_spans),
            "window_source":        batch["context_text"],
            "context_hash":         ctx_hash,
            "context_token_count":  batch["context_token_count"],
            "context_chunk_ids":    batch["context_chunk_ids"],
            "source_pages":         batch["source_pages"],
            "batch_index":          batch_idx,
            "num_batches":          num_batches,
            "document_title":       data["document_title"],
            "document_year":        data["document_year"],
            "section_id":           data["section_id"],
            "section_title":        data["section_title"],
            "bps_url":              data["bps_url"],
            "parse_status":         parse_status,
            "prompt_tokens":        usage["prompt_tokens"],
            "completion_tokens":    usage["completion_tokens"],
            "cached_tokens":        usage["cached_tokens"],
            "generation_model":     CONFIG["llm_model"],
            "generation_timestamp": GENERATION_TIMESTAMP,
        })
    return records, skipped

### g. Assembly & Export

In [12]:
def run_assembly(total_documents, total_sections_raw, total_sections_selected,
                 filter_stats, total_raw_records, total_failures, total_skipped,
                 agg_tokens, parse_status_dist, task_dist):
    # Hitung persentase token yang di-hit dari cache Groq (mengurangi biaya efektif)
    cache_hit = round(agg_tokens["cached_tokens"] / agg_tokens["prompt_tokens"] * 100, 2) \
        if agg_tokens["prompt_tokens"] else 0
 
    # Kumpulkan semua statistik ke satu dict
    stats = {
        "dataset_version":           CONFIG["dataset_version"],
        "generation_model":          CONFIG["llm_model"],
        "generation_timestamp":      GENERATION_TIMESTAMP,
        "total_documents":           total_documents,
        "total_sections_raw":        total_sections_raw,
        "total_sections_selected":   total_sections_selected,
        # Distribusi alasan filter (bad_source, too_short, all_non_text, noise_title)
        "filter_statistics":         filter_stats,
        "total_raw_records":         total_raw_records,
        "total_failures":            total_failures,
        "total_skipped_items":       total_skipped,
        # Agregat token lintas seluruh request
        "token_usage":               agg_tokens,
        "cache_hit_rate_pct":        cache_hit,
        # Distribusi status parse (direct, fence_stripped, extracted, failed, api_error)
        "parse_status_distribution": parse_status_dist,
        # Distribusi tipe tugas (qa/information_retrieval, summary/document_overview, dll.)
        "task_distribution_raw":     task_dist,
    }
 
    with open(f"{OUT}/generation_statistics.json", "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)
 
    # Cetak ringkasan ke layar (untuk output sel Kaggle)
    print("=" * 60)
    print("05a — GENERATION SUMMARY")
    print("=" * 60)
    print(f"Dataset Version        : {CONFIG['dataset_version']}")
    print(f"Model                  : {CONFIG['llm_model']}")
    print(f"Total Documents        : {total_documents}")
    print(f"Sections Raw           : {total_sections_raw}")
    print(f"Sections Selected      : {total_sections_selected}")
    print(f"Filter Stats           : {filter_stats}")
    print(f"Total Raw Records      : {total_raw_records}")
    print(f"Total Failures         : {total_failures}")
    print(f"Total Skipped Items    : {total_skipped}")
    print(f"\nToken Usage:")
    print(f"  Prompt Tokens        : {agg_tokens['prompt_tokens']:,}")
    print(f"  Cached Tokens        : {agg_tokens['cached_tokens']:,}")
    print(f"  Completion Tokens    : {agg_tokens['completion_tokens']:,}")
    print(f"  Total Tokens         : {agg_tokens['total_tokens']:,}")
    print(f"  Cache Hit Rate       : {cache_hit:.2f}%")
    print(f"\nParse Status Distribution:")
    for k, v in sorted(parse_status_dist.items()):
        print(f"  {k}: {v}")
    print(f"\nTask Distribution (raw):")
    for k, v in sorted(task_dist.items()):
        print(f"  {k}: {v}")

### h. Reset output

In [13]:
import shutil
shutil.rmtree(CONFIG["output_dir"], ignore_errors=True)
os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(f"{CONFIG['output_dir']}/logs", exist_ok=True)
print("output_05a dibersihkan, mulai run baru dari nol")

output_05a dibersihkan, mulai run baru dari nol


## 3. Main Program

In [14]:
# Persiapan data
chunks          = load_chunks(CONFIG["input_dir"])
all_sections    = reconstruct_sections(chunks)
selected, filter_stats = select_sections(all_sections)
if CONFIG["test_mode"] and CONFIG.get("max_documents_test"):
    doc_order = []
    for sec_data in all_sections.values():
        if sec_data["document_title"] not in doc_order:
            doc_order.append(sec_data["document_title"])
    allowed_docs = set(doc_order[:CONFIG["max_documents_test"]])
    selected = {k: v for k, v in selected.items() if v["document_title"] in allowed_docs}
completed_units = load_completed_units()
 
# Hitung statistik awal
total_documents         = len(set(d["document_title"] for d in selected.values()))
total_sections_raw      = len(all_sections)
total_sections_selected = len(selected)
sections_to_process     = list(selected.items())
 
gen_log.info(f"raw={total_sections_raw} selected={total_sections_selected} done={len(completed_units)}")
print(f"Sections raw            : {total_sections_raw}")
print(f"Sections selected       : {total_sections_selected}")
print(f"Completed units (resume): {len(completed_units)}")
 
# Inisialisasi akumulator
agg_tokens         = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "cached_tokens": 0}
total_raw_records  = 0
total_failures     = 0
total_skipped      = 0
parse_status_dist  = {}
task_dist          = {}
global_batch_count = 0
_last_records      = []

Sections raw            : 7535
Sections selected       : 1069
Completed units (resume): 0


### i. Diagnostik : estimasi jumlah batch full production

In [15]:
diag_total_batches = 0
diag_n_pairs_dist = {2: 0, 4: 0, 6: 0}
for sec_data in selected.values():
    bs = batch_section(sec_data["section_chunks"], CONFIG["max_tokens_per_context"], CONFIG["overlap_tokens"])
    diag_total_batches += len(bs)
    for b in bs:
        diag_n_pairs_dist[get_n_pairs(b["context_token_count"])] += 1
 
print(f"Sections selected      : {len(selected)}")
print(f"Total batches (full)   : {diag_total_batches}")
print(f"Avg batch/section      : {diag_total_batches/len(selected):.2f}")
print(f"n_pairs distribution   : {diag_n_pairs_dist}")
print(f"Estimasi total call    : {diag_total_batches * 5} (4 QA + 1 summary per batch)")

Sections selected      : 1069
Total batches (full)   : 2248
Avg batch/section      : 2.10
n_pairs distribution   : {2: 995, 4: 1253, 6: 0}
Estimasi total call    : 11240 (4 QA + 1 summary per batch)


### j. Jalankan pipeline

In [16]:
# Loop utama
async def run_pipeline():
    global total_raw_records, total_failures, total_skipped, global_batch_count, _last_records
 
    try:
        await _run_pipeline_inner()
    except BudgetExhausted as e:
        print(f"\n[STOP] Saldo DeepSeek habis, pipeline dihentikan lebih awal: {e}")
        print(f"[STOP] Progres tersimpan: {total_raw_records} record sudah ter-generate sebelum berhenti.")
 
 
async def _run_pipeline_inner():
    global total_raw_records, total_failures, total_skipped, global_batch_count, _last_records
 
    for sec_key, data in tqdm(sections_to_process, desc="Sections"):
        print(f"\n>>> [{data['document_title'][:45]}] {data['section_title'][:45]} ({data['section_id']})")
        batches     = batch_section(data["section_chunks"], CONFIG["max_tokens_per_context"], CONFIG["overlap_tokens"])
        num_batches = len(batches)
 
        batches_to_run = batches[:1] if CONFIG["test_mode"] else batches
        for batch_idx, batch in enumerate(tqdm(batches_to_run, desc="Batches", leave=False)):
            ctx_hash    = hashlib.sha256(batch["context_text"].encode()).hexdigest()
            batch_label = f"{data['section_id']}|b{batch_idx}"
            ctx_tokens  = batch["context_token_count"]
 
            pending_qa = [
                qa_cfg for qa_cfg in QA_BATCHES
                if unit_key(data["document_title"], data["section_id"], batch_idx, qa_cfg["generation_focus"])
                not in completed_units
            ]
 
            if pending_qa:
                qa_results, n_pairs = await generate_all_qa_parallel(batch["context_text"], ctx_tokens, batch_label)
                await asyncio.sleep(CONFIG["request_delay"])
 
                for i, qa_cfg in enumerate(QA_BATCHES):
                    uk = unit_key(data["document_title"], data["section_id"], batch_idx, qa_cfg["generation_focus"])
                    if uk in completed_units:
                        continue
 
                    result = qa_results[i]
                    if isinstance(result, Exception):
                        total_failures += 1
                        print(f"  [ERR ] {batch_label}|{qa_cfg['generation_focus']}: {str(result)[:100]}")
                        append_jsonl(FAILURES_JSONL, [{
                            "section_id": data["section_id"], "batch_index": batch_idx,
                            "generation_focus": qa_cfg["generation_focus"],
                            "error": str(result), "timestamp": GENERATION_TIMESTAMP,
                        }])
                        mark_unit(uk); continue
 
                    parsed, parse_status, raw, usage = result
                    for k in agg_tokens: agg_tokens[k] += usage[k]
                    parse_status_dist[parse_status] = parse_status_dist.get(parse_status, 0) + 1
 
                    if parsed is None:
                        total_failures += 1
                        print(f"  [FAIL] {batch_label}|{qa_cfg['generation_focus']} parse={parse_status}")
                        append_jsonl(FAILURES_JSONL, [{
                            "section_id": data["section_id"], "batch_index": batch_idx,
                            "generation_focus": qa_cfg["generation_focus"], "gen_type": "qa",
                            "context_hash": ctx_hash, "parse_status": parse_status,
                            "raw_output": raw, "timestamp": GENERATION_TIMESTAMP,
                        }])
                        mark_unit(uk); continue
 
                    new_records, skipped = process_batch(
                        batch, batch_idx, num_batches, data, ctx_hash,
                        parsed, parse_status, qa_cfg["generation_focus"], usage,
                    )
                    if new_records:
                        append_jsonl(RAW_JSONL, new_records)
                        total_raw_records += len(new_records)
                        for rec in new_records:
                            k = f"{rec['task_type']}/{rec['subtask_type']}"
                            task_dist[k] = task_dist.get(k, 0) + 1
                        gen_log.info(f"{batch_label}|{qa_cfg['generation_focus']} → {len(new_records)} records")
                        _last_records = new_records
                        print(f"  [QA:{qa_cfg['pattern'][:8]}] {batch_label} +{len(new_records)} | tok={usage['total_tokens']} | total={total_raw_records}")
                    total_skipped += skipped
                    mark_unit(uk)
 
            uk_sum = unit_key(data["document_title"], data["section_id"], batch_idx, "summary")
            if uk_sum not in completed_units:
                parsed, parse_status, raw, usage = generate_summary(
                    batch["context_text"], data["section_title"], label=f"{batch_label}|summary",
                )
                await asyncio.sleep(CONFIG["request_delay"])
 
                for k in agg_tokens: agg_tokens[k] += usage[k]
                parse_status_dist[parse_status] = parse_status_dist.get(parse_status, 0) + 1
 
                if parsed is None:
                    total_failures += 1
                    print(f"  [FAIL] {batch_label}|summary parse={parse_status}")
                    append_jsonl(FAILURES_JSONL, [{
                        "section_id": data["section_id"], "batch_index": batch_idx,
                        "generation_focus": "summary", "gen_type": "summary",
                        "context_hash": ctx_hash, "parse_status": parse_status,
                        "raw_output": raw, "timestamp": GENERATION_TIMESTAMP,
                    }])
                else:
                    new_records, skipped = process_batch(
                        batch, batch_idx, num_batches, data, ctx_hash,
                        parsed, parse_status, "summary", usage,
                    )
                    if new_records:
                        append_jsonl(RAW_JSONL, new_records)
                        total_raw_records += len(new_records)
                        for rec in new_records:
                            k = f"{rec['task_type']}/{rec['subtask_type']}"
                            task_dist[k] = task_dist.get(k, 0) + 1
                        gen_log.info(f"{batch_label}|summary → {len(new_records)} records")
                        print(f"  [SUM] {batch_label} +{len(new_records)} | tok={usage['total_tokens']} | total={total_raw_records}")
                    total_skipped += skipped
                mark_unit(uk_sum)
 
            global_batch_count += 1
            if global_batch_count % CONFIG["preview_interval"] == 0 and _last_records:
                print_preview(_last_records, batch_label)
            if global_batch_count % CONFIG["checkpoint_interval"] == 0:
                avg_tok = agg_tokens["total_tokens"] / global_batch_count
                msg = f"[CHECKPOINT batch={global_batch_count}] records={total_raw_records} failures={total_failures} avg_tok={avg_tok:.0f}"
                gen_log.info(msg)
                print(f"  {msg}")
 
await run_pipeline()
 
run_assembly(
    total_documents, total_sections_raw, total_sections_selected,
    filter_stats, total_raw_records, total_failures, total_skipped,
    agg_tokens, parse_status_dist, task_dist,
)

Sections:   0%|          | 0/1069 [00:00<?, ?it/s]


>>> [Analisis Petty Corruption Hasil SPAK 2020–202] 5.3 Membayar Melebihi Ketentuan (sec_101)


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

  [QA:literal_] sec_101|b0 +3 | tok=5802 | total=3
  [QA:concept_] sec_101|b0 +3 | tok=5901 | total=6
  [QA:evidence] sec_101|b0 +3 | tok=5965 | total=9
  [QA:analytic] sec_101|b0 +2 | tok=6733 | total=11
  [QA:literal_] sec_101|b1 +3 | tok=5724 | total=14
  [QA:concept_] sec_101|b1 +2 | tok=5638 | total=16
  [QA:evidence] sec_101|b1 +4 | tok=6217 | total=20
  [QA:analytic] sec_101|b1 +2 | tok=6494 | total=22
  [SUM] sec_101|b1 +1 | tok=4082 | total=23
  [QA:literal_] sec_101|b2 +3 | tok=5764 | total=26
  [QA:concept_] sec_101|b2 +3 | tok=5998 | total=29
  [QA:evidence] sec_101|b2 +4 | tok=6220 | total=33
  [QA:analytic] sec_101|b2 +4 | tok=6134 | total=37
  [SUM] sec_101|b2 +1 | tok=4236 | total=38
  [QA:literal_] sec_101|b3 +2 | tok=3905 | total=40
  [QA:concept_] sec_101|b3 +2 | tok=3819 | total=42
  [QA:evidence] sec_101|b3 +2 | tok=4038 | total=44
  [QA:analytic] sec_101|b3 +2 | tok=4294 | total=46
  [SUM] sec_101|b3 +1 | tok=2838 | total=47

>>> [Analisis Petty Corruption Hasil S

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

...

  [QA:literal_] sec_169|b0 +4 | tok=6917 | total=11836
  [QA:concept_] sec_169|b0 +4 | tok=7174 | total=11840
  [QA:analytic] sec_169|b0 +4 | tok=7554 | total=11844
  [SUM] sec_169|b0 +1 | tok=5590 | total=11845
  [QA:literal_] sec_169|b1 +3 | tok=7199 | total=11848
  [QA:concept_] sec_169|b1 +4 | tok=6952 | total=11852
  [QA:analytic] sec_169|b1 +4 | tok=7421 | total=11856
  [SUM] sec_169|b1 +1 | tok=5653 | total=11857
  [QA:literal_] sec_169|b2 +4 | tok=6995 | total=11861
  [QA:concept_] sec_169|b2 +4 | tok=7399 | total=11865
  [QA:analytic] sec_169|b2 +4 | tok=7378 | total=11869
  [SUM] sec_169|b2 +1 | tok=5557 | total=11870
  [QA:literal_] sec_169|b3 +4 | tok=6779 | total=11874
  [QA:concept_] sec_169|b3 +4 | tok=6718 | total=11878
  [QA:evidence] sec_169|b3 +4 | tok=6923 | total=11882
  [QA:analytic] sec_169|b3 +4 | tok=7797 | total=11886
  [SUM] sec_169|b3 +1 | tok=5334 | total=11887
  [QA:literal_] sec_169|b4 +2 | tok=6739 | total=11889
  [QA:concept_] sec_169|b4 +3 | tok=6576 |

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

  [QA:literal_] sec_236|b0 +4 | tok=6904 | total=12003
  [QA:concept_] sec_236|b0 +4 | tok=6851 | total=12007
  [QA:evidence] sec_236|b0 +4 | tok=7129 | total=12011
  [QA:analytic] sec_236|b0 +4 | tok=7414 | total=12015
  [SUM] sec_236|b0 +1 | tok=5348 | total=12016
  [QA:literal_] sec_236|b1 +4 | tok=7416 | total=12020
  [QA:concept_] sec_236|b1 +4 | tok=7374 | total=12024
  [QA:evidence] sec_236|b1 +2 | tok=7691 | total=12026
  [QA:analytic] sec_236|b1 +4 | tok=7904 | total=12030
  [SUM] sec_236|b1 +1 | tok=5785 | total=12031
  [QA:literal_] sec_236|b2 +4 | tok=6550 | total=12035
  [QA:concept_] sec_236|b2 +4 | tok=6168 | total=12039
  [QA:evidence] sec_236|b2 +4 | tok=6798 | total=12043
  [QA:analytic] sec_236|b2 +4 | tok=6679 | total=12047
  [SUM] sec_236|b2 +1 | tok=4627 | total=12048
  [QA:literal_] sec_236|b3 +4 | tok=6824 | total=12052
  [QA:concept_] sec_236|b3 +3 | tok=6621 | total=12055
  [QA:evidence] sec_236|b3 +4 | tok=7114 | total=12059
  [QA:analytic] sec_236|b3 +2 | to

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

  [QA:literal_] sec_132|b0 +4 | tok=6907 | total=12159
  [QA:concept_] sec_132|b0 +2 | tok=6758 | total=12161
  [QA:analytic] sec_132|b0 +4 | tok=7050 | total=12165
  [QA:literal_] sec_132|b1 +4 | tok=6903 | total=12169
  [QA:concept_] sec_132|b1 +4 | tok=6983 | total=12173
  [QA:evidence] sec_132|b1 +4 | tok=7408 | total=12177
  [QA:analytic] sec_132|b1 +4 | tok=7685 | total=12181
  [SUM] sec_132|b1 +1 | tok=5238 | total=12182
  [QA:literal_] sec_132|b2 +4 | tok=7179 | total=12186
  [QA:concept_] sec_132|b2 +3 | tok=6917 | total=12189
  [QA:evidence] sec_132|b2 +4 | tok=7239 | total=12193
  [QA:analytic] sec_132|b2 +4 | tok=7341 | total=12197
  [QA:literal_] sec_132|b3 +4 | tok=6764 | total=12201
  [QA:concept_] sec_132|b3 +4 | tok=6894 | total=12205
  [QA:evidence] sec_132|b3 +4 | tok=6963 | total=12209
  [QA:analytic] sec_132|b3 +4 | tok=7743 | total=12213
  [SUM] sec_132|b3 +1 | tok=5237 | total=12214

────────────────────────────────────────────────────────────
PREVIEW  sec_132|b3